In [67]:
import time
import requests
import pandas as pd
import csv

In [100]:
HEADERS = {"User-Agent": "BookDatasetBuilder/1.0 (contact: you@example.com)"}

# Mapea la lista de géneros -> el "subject" que usa Open Library en la URL.

GENRE_TO_SUBJECT = {
    "Fantasy": "fantasy",
    "Science fiction": "science_fiction",
    "Mystery": "mystery",
    "Romance": "romance",
    "Horror": "horror",
    "Historical fiction": "historical_fiction",
    "Young adult": "young_adult",
    "Non fiction": "nonfiction",
}

def fetch_subject_works(subject_slug: str, limit: int = 200, sleep_s: float = 0.2) -> list[dict]:
    """
    Open Library Subjects API:
    GET /subjects/{subject}.json?limit=...
    Devuelve obras asociadas al subject.  :contentReference[oaicite:1]{index=1}
    """
    url = f"https://openlibrary.org/subjects/{subject_slug}.json"
    params = {"limit": limit, "details": "true"}  # details añade info útil cuando esté disponible
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    time.sleep(sleep_s)
    return data.get("works", [])

def best_year(work: dict):
    # subjects API suele traer first_publish_year o first_publish_date
    y = work.get("first_publish_year")
    if isinstance(y, int):
        return y
    d = work.get("first_publish_date")
    if isinstance(d, str) and len(d) >= 4 and d[:4].isdigit():
        return int(d[:4])
    return None

def best_author(work: dict):
    authors = work.get("authors") or []
    if authors and isinstance(authors, list):
        name = authors[0].get("name")
        return name
    return None


def f_synopsis(work_key: str, sleep_s: float = 1, delay: int = 4):
    """
    Obtiene la sinopsis (description) desde el endpoint de la obra:
    https://openlibrary.org{work_key}.json

    Nota: 'description' puede venir como string o como dict {'value': '...'}.
    """
    if not work_key:
        return None

    url = f"https://openlibrary.org{work_key}.json"
    
    while True:
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            data = r.json()

            description = data.get("description")
            if isinstance(description, str):
                synopsis = description.strip()
            elif isinstance(description, dict):
                synopsis = (description.get("value") or "").strip()
            else:
                synopsis = None

            time.sleep(sleep_s)
            return synopsis

        except requests.exceptions.HTTPError:
            if r.status_code >= 500:
                print(f"Error {r.status_code} (servidor). Reintentando en {delay}s...")
            else:
                # 4xx → no reintentamos
                raise
        
        except requests.exceptions.RequestException:
            print(f"Error de conexión. Reintentando en {delay}s...")
        
        time.sleep(min(2 ** attempt, 60))
        attempt += 1


## AQUÍ ES DONDE TENGO QUE AÑADIR PARA LA SINOPSIS ##

######################################################################################################




def build_dataset(per_genre: int = 200) -> pd.DataFrame:
    rows = []
    for genre, subject in GENRE_TO_SUBJECT.items():
        works = fetch_subject_works(subject, limit=per_genre)
        for w in works[:per_genre]:
            rows.append({
                "title": w.get("title"),
                "author": best_author(w),
                "year": best_year(w),
                "genre": genre,
                "openlibrary_work_key": w.get("key")
            })

    df = pd.DataFrame(rows)
    return df

def basic_clean(df) -> pd.DataFrame:

    # Limpieza básica
    for c in ["title", "author", "genre"]:
        df[c] = df[c].astype("string").str.strip()

    df = df[df["title"].notna() & (df["title"].str.len() > 0)].copy()

    # Deduplicación simple (título + autor + año)
    df["dedupe"] = (
        df["title"].fillna("") + "||" +
        df["author"].fillna("") + "||" +
        df["year"].fillna(-1).astype("int64").astype("string")
    )
    df = df.drop_duplicates("dedupe").drop(columns=["dedupe"])

    return df


df = build_dataset(per_genre=200)  # ajustar a 50/200/1000 según se quiera. ¡Es por género!
df.to_csv("books_multi_genre.csv", index=False, encoding="utf-8")
# df = basic_clean(df)
print("Filas:", len(df))
print(df.head(10).to_string(index=False))

Filas: 1600
                           title                 author  year   genre openlibrary_work_key
Alice's Adventures in Wonderland          Lewis Carroll  1865 Fantasy     /works/OL138052W
      The Wonderful Wizard of Oz          L. Frank Baum  1899 Fantasy      /works/OL18417W
                 Treasure Island Robert Louis Stevenson  1880 Fantasy      /works/OL24034W
              Gulliver's Travels         Jonathan Swift  1726 Fantasy      /works/OL20600W
       A Midsummer Night's Dream    William Shakespeare  1600 Fantasy     /works/OL259010W
                      The Prince    Niccolò Machiavelli  1515 Fantasy    /works/OL1089297W
       Through the Looking-Glass          Lewis Carroll  1865 Fantasy     /works/OL151406W
         The Wind in the Willows        Kenneth Grahame  1908 Fantasy   /works/OL28570037W
            Five Children and It           Edith Nesbit  1905 Fantasy      /works/OL99499W
     The Princess and the Goblin       George MacDonald  1872 Fantasy      /wo

In [49]:
rows = []
contador = 1
for r in df.itertuples():
    print(f"Recogiendo sinopsis del libro número {contador}: {r.title}")
    synopsis = f_synopsis(r.openlibrary_work_key)
    rows.append({
        "openlibrary_work_key": r.openlibrary_work_key,
        "synopsis": synopsis
    })
    contador+=1

df_synopsis = pd.DataFrame(rows)
df_synopsis

Recogiendo sinopsis del libro número 1: Alice's Adventures in Wonderland
Recogiendo sinopsis del libro número 2: The Wonderful Wizard of Oz
Recogiendo sinopsis del libro número 3: Treasure Island
Recogiendo sinopsis del libro número 4: Gulliver's Travels
Recogiendo sinopsis del libro número 5: A Midsummer Night's Dream
Recogiendo sinopsis del libro número 6: The Prince
Recogiendo sinopsis del libro número 7: Through the Looking-Glass
Recogiendo sinopsis del libro número 8: The Wind in the Willows
Recogiendo sinopsis del libro número 9: Five Children and It
Recogiendo sinopsis del libro número 10: The Princess and the Goblin
Recogiendo sinopsis del libro número 11: The Lost World
Recogiendo sinopsis del libro número 12: The Marvelous Land of Oz
Recogiendo sinopsis del libro número 13: Ozma of Oz
Recogiendo sinopsis del libro número 14: Le petit prince
Recogiendo sinopsis del libro número 15: The Emerald City of Oz
Recogiendo sinopsis del libro número 16: Dorothy and the Wizard in Oz
Rec

,openlibrary_work_key,synopsis
0,/works/OL138052W,Alice's Adventures in Wonderland (commonly Ali...
1,/works/OL18417W,"Over a century after its initial publication, ..."
2,/works/OL24034W,Traditionally considered a coming-of-age story...
3,/works/OL20600W,A parody of traveler’s tales and a satire of h...
4,/works/OL259010W,One night two young couples run into an enchan...
...,...,...
1595,/works/OL2147971W,When Fanny Trollope set sail for America in 18...
1596,/works/OL1496762W,None
1597,/works/OL65487W,Runaway best-selling PC hardware book of all t...
1598,/works/OL53604W,Helen Keller graduated cum laude from Radcliff...


In [96]:
# Eliminamos duplicados
df_synopsis_clean = df_synopsis.drop_duplicates("openlibrary_work_key")
print("Filas:", len(df_synopsis_clean))
print(df_synopsis_clean.head(10).to_string(index=False))

Filas: 1461
openlibrary_work_key                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [102]:
# Por último unimos los dataframes para trabajar con un único dataframe en el que se incluya la sinopsis y generamos el CSV
df_final = df.merge(df_synopsis_clean, how="left", on="openlibrary_work_key")
# df_final.to_csv("books_with_synopsis.csv", index=False, encoding="utf-8")
print("Filas:", len(df_final))
print(df_final.head(10).to_string(index=False))

Filas: 1600
                           title                 author  year   genre openlibrary_work_key                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [84]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600 entries, 0 to 1599
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   title                 1600 non-null   object
 1   author                1600 non-null   object
 2   year                  1600 non-null   int64 
 3   genre                 1600 non-null   object
 4   openlibrary_work_key  1600 non-null   object
 5   synopsis              1512 non-null   object
dtypes: int64(1), object(5)
memory usage: 75.1+ KB


In [85]:
df_final['genre'].value_counts(dropna=False)

genre
Fantasy               200
Science fiction       200
Mystery               200
Romance               200
Horror                200
Historical fiction    200
Young adult           200
Non fiction           200
Name: count, dtype: int64

In [86]:
df_clean = basic_clean(df_final)
print("Filas:", len(df_clean))
df_clean.head(10)

Filas: 1461


,title,author,year,genre,openlibrary_work_key,synopsis
0,Alice's Adventures in Wonderland,Lewis Carroll,1865,Fantasy,/works/OL138052W,Alice's Adventures in Wonderland (commonly Ali...
1,The Wonderful Wizard of Oz,L. Frank Baum,1899,Fantasy,/works/OL18417W,"Over a century after its initial publication, ..."
2,Treasure Island,Robert Louis Stevenson,1880,Fantasy,/works/OL24034W,Traditionally considered a coming-of-age story...
3,Gulliver's Travels,Jonathan Swift,1726,Fantasy,/works/OL20600W,A parody of traveler’s tales and a satire of h...
4,A Midsummer Night's Dream,William Shakespeare,1600,Fantasy,/works/OL259010W,One night two young couples run into an enchan...
5,The Prince,Niccolò Machiavelli,1515,Fantasy,/works/OL1089297W,The Prince (Italian: Il Principe [il ˈprintʃip...
6,Through the Looking-Glass,Lewis Carroll,1865,Fantasy,/works/OL151406W,"*Through the Looking-Glass, and What Alice Fou..."
7,The Wind in the Willows,Kenneth Grahame,1908,Fantasy,/works/OL28570037W,"The adventures of four amiable animals, Rat, T..."
8,Five Children and It,Edith Nesbit,1905,Fantasy,/works/OL99499W,Haven't you ever thought what you would wish f...
9,The Princess and the Goblin,George MacDonald,1872,Fantasy,/works/OL15449W,There was once a little princess whose father ...


In [87]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1461 entries, 0 to 1599
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   title                 1461 non-null   string
 1   author                1461 non-null   string
 2   year                  1461 non-null   int64 
 3   genre                 1461 non-null   string
 4   openlibrary_work_key  1461 non-null   object
 5   synopsis              1373 non-null   object
dtypes: int64(1), object(2), string(3)
memory usage: 79.9+ KB


In [88]:
df_clean['genre'].value_counts(dropna=False)

genre
Fantasy               200
Mystery               195
Non fiction           194
Romance               188
Young adult           180
Horror                177
Historical fiction    164
Science fiction       163
Name: count, dtype: Int64

In [104]:
df_final['year'].min()

0

In [89]:
df_clean['year'].min()

0

In [ ]:
# DF original
print(f"El intervalo de años de publicación va de {df_final['year'].min()} a {df_final['year'].max()}")

El intervalo de años de publicación va de 0 a 2025


In [ ]:
# DF sin duplicados
print(f"El intervalo de años de publicación va de {df_clean['year'].min()} a {df_clean['year'].max()}")

El intervalo de años de publicación va de 0 a 2025


In [106]:
# DF original
found = False

for row in df_final.itertuples(): 
    if row.year == 0:
        print(f'El título: {row.title} tiene un cero ({row.year}) como año de publicación.')
        found = True

if not found: 
    print('Todos los libros tienen año de publicación')
    

El título: The Invisible Man tiene un cero (0) como año de publicación.
El título: The War of the Worlds tiene un cero (0) como año de publicación.


In [92]:
# DF sin duplicados
found_clean = False

for row in df_clean.itertuples(): 
    if row.year == 0:
        print(f'El título: {row.title} tiene un cero ({row.year}) como año de publicación.')
        found_clean = True

if not found_clean: 
    print('Todos los libros tienen año de publicación')


El título: The Invisible Man tiene un cero (0) como año de publicación.
El título: The War of the Worlds tiene un cero (0) como año de publicación.


Cambiamos por los años que corresponde: 

- The Invisible Man -> 1897
- The War of the Worlds -> 1898

In [ ]:
#DF original
df_final.loc[df_final['title'] == 'The Invisible Man', 'year'] = 1897
df_final.loc[df_final['title'] == 'The War of the Worlds', 'year'] = 1898

# Comprobación del DF original
found = False

for row in df_final.itertuples(): 
    if row.year == 0:
        print(f'El título: {row.title} tiene un cero ({row.year}) como año de publicación.')
        found = True

if not found: 
    print('Todos los libros tienen año de publicación')

Todos los libros tienen año de publicación


In [ ]:
# DF sin duplicados
df_clean.loc[df_clean['title'] == 'The Invisible Man', 'year'] = 1897
df_clean.loc[df_clean['title'] == 'The War of the Worlds', 'year'] = 1898

# Comprobación de DF sin duplicados
found_clean = False

for row in df_clean.itertuples(): 
    if row.year == 0:
        print(f'El título: {row.title} tiene un cero ({row.year}) como año de publicación.')
        found_clean = True

if not found_clean: 
    print('Todos los libros tienen año de publicación')

Todos los libros tienen año de publicación


In [114]:
# Definir los límites
bins = [
    1400, 1500, 1600, 1700, 1800,
    1850, 1900, 1950, 1970, 1990,
    2000, 2010, 2020, 2027
]

labels = [
    "1400-1499",
    "1500-1599",
    "1600-1699",
    "1700-1799",
    "1800-1849",
    "1850-1899",
    "1900-1949",
    "1950-1969",
    "1970-1989",
    "1990-1999",
    "2000-2009",
    "2010-2019",
    "2020-2026"
]


# DF original 
df_final["periodo"] = pd.cut(
    df_final["year"],
    bins=bins,
    labels=labels,
    right=False
)


# DF sin duplicados
df_clean["periodo"] = pd.cut(
    df_clean["year"],
    bins=bins,
    labels=labels,
    right=False
)


In [115]:
df_final.head(3)

,title,author,year,genre,openlibrary_work_key,synopsis,periodo
0,Alice's Adventures in Wonderland,Lewis Carroll,1865,Fantasy,/works/OL138052W,Alice's Adventures in Wonderland (commonly Ali...,1850-1899
1,The Wonderful Wizard of Oz,L. Frank Baum,1899,Fantasy,/works/OL18417W,"Over a century after its initial publication, ...",1850-1899
2,Treasure Island,Robert Louis Stevenson,1880,Fantasy,/works/OL24034W,Traditionally considered a coming-of-age story...,1850-1899


In [117]:
df_clean.tail(3)

,title,author,year,genre,openlibrary_work_key,synopsis,periodo
1597,Upgrading and repairing PCs,Scott Mueller,1988,Non fiction,/works/OL65487W,Runaway best-selling PC hardware book of all t...,1970-1989
1598,The story of my life,Helen Keller,1903,Non fiction,/works/OL53604W,Helen Keller graduated cum laude from Radcliff...,1900-1949
1599,The Power of Now,Eckhart Tolle,1997,Non fiction,/works/OL5727686W,Eckhart Tolle has emerged as one of today's mo...,1990-1999


In [120]:
# DF original
# Una vez limpio y tratado el dataframe volvemos a generarlo en CSV
df_final.to_csv(
    "books_original.csv",
    index=False,
    encoding="utf-8",
    sep=",",
    quoting=csv.QUOTE_ALL,
    doublequote=True,
    quotechar='"',
    escapechar="\\",
    lineterminator="\n")
print("Filas:", len(df_final))
df_final.head(10)

Filas: 1600


,title,author,year,genre,openlibrary_work_key,synopsis,periodo
0,Alice's Adventures in Wonderland,Lewis Carroll,1865,Fantasy,/works/OL138052W,Alice's Adventures in Wonderland (commonly Ali...,1850-1899
1,The Wonderful Wizard of Oz,L. Frank Baum,1899,Fantasy,/works/OL18417W,"Over a century after its initial publication, ...",1850-1899
2,Treasure Island,Robert Louis Stevenson,1880,Fantasy,/works/OL24034W,Traditionally considered a coming-of-age story...,1850-1899
3,Gulliver's Travels,Jonathan Swift,1726,Fantasy,/works/OL20600W,A parody of traveler’s tales and a satire of h...,1700-1799
4,A Midsummer Night's Dream,William Shakespeare,1600,Fantasy,/works/OL259010W,One night two young couples run into an enchan...,1600-1699
5,The Prince,Niccolò Machiavelli,1515,Fantasy,/works/OL1089297W,The Prince (Italian: Il Principe [il ˈprintʃip...,1500-1599
6,Through the Looking-Glass,Lewis Carroll,1865,Fantasy,/works/OL151406W,"*Through the Looking-Glass, and What Alice Fou...",1850-1899
7,The Wind in the Willows,Kenneth Grahame,1908,Fantasy,/works/OL28570037W,"The adventures of four amiable animals, Rat, T...",1900-1949
8,Five Children and It,Edith Nesbit,1905,Fantasy,/works/OL99499W,Haven't you ever thought what you would wish f...,1900-1949
9,The Princess and the Goblin,George MacDonald,1872,Fantasy,/works/OL15449W,There was once a little princess whose father ...,1850-1899


In [121]:
# DF sin duplicados
# Una vez limpio y tratado el dataframe volvemos a generarlo en CSV
df_clean.to_csv(
    "books_clean.csv",
    index=False,
    encoding="utf-8",
    sep=",",
    quoting=csv.QUOTE_ALL,
    doublequote=True,
    quotechar='"',
    escapechar="\\",
    lineterminator="\n")
print("Filas:", len(df_clean))
df_clean.tail(10)

Filas: 1461


,title,author,year,genre,openlibrary_work_key,synopsis,periodo
1590,Las venas abiertas de América Latina,Eduardo Galeano,1971,Non fiction,/works/OL698173W,"Since its U.S. debut almost fifty years ago, t...",1970-1989
1591,"Behind the scenes. By Elizabeth Keckley. Or, T...",Elizabeth Keckley,1868,Non fiction,/works/OL4968216W,A former slave's intimate memoir of the Lincol...,1850-1899
1592,Personal Memoirs of U. S. Grant (2 volumes in 1),Ulysses S. Grant,1894,Non fiction,/works/OL1928235W,"Tracing his ancestry, Grant gives insight into...",1850-1899
1593,The history of England from the accession of J...,Thomas Babington Macaulay,1849,Non fiction,/works/OL8522726W,While instituting the English education system...,1800-1849
1594,The science of being great,Wallace D. Wattles,1911,Non fiction,/works/OL3792048W,"Newly rediscovered by fans of The Secret, the ...",1900-1949
1595,Domestic manners of the Americans,Frances Milton Trollope,1832,Non fiction,/works/OL2147971W,When Fanny Trollope set sail for America in 18...,1800-1849
1596,Les tragiques,Agrippa d' Aubigné,1800,Non fiction,/works/OL1496762W,None,1800-1849
1597,Upgrading and repairing PCs,Scott Mueller,1988,Non fiction,/works/OL65487W,Runaway best-selling PC hardware book of all t...,1970-1989
1598,The story of my life,Helen Keller,1903,Non fiction,/works/OL53604W,Helen Keller graduated cum laude from Radcliff...,1900-1949
1599,The Power of Now,Eckhart Tolle,1997,Non fiction,/works/OL5727686W,Eckhart Tolle has emerged as one of today's mo...,1990-1999


In [111]:
df_title_syn = df_clean[['title', 'synopsis']]
df_title_syn.head(2)

,title,synopsis
0,Alice's Adventures in Wonderland,Alice's Adventures in Wonderland (commonly Ali...
1,The Wonderful Wizard of Oz,"Over a century after its initial publication, ..."


In [122]:
df_title_syn.to_csv(
    "books_title_synopsis.csv",
    index=False,
    encoding="utf-8",
    sep=",",
    quoting=csv.QUOTE_ALL,
    doublequote=True,
    quotechar='"',
    escapechar="\\",
    lineterminator="\n")
print("Filas:", len(df_title_syn))
df_title_syn.head(10)

Filas: 1461


,title,synopsis
0,Alice's Adventures in Wonderland,Alice's Adventures in Wonderland (commonly Ali...
1,The Wonderful Wizard of Oz,"Over a century after its initial publication, ..."
2,Treasure Island,Traditionally considered a coming-of-age story...
3,Gulliver's Travels,A parody of traveler’s tales and a satire of h...
4,A Midsummer Night's Dream,One night two young couples run into an enchan...
5,The Prince,The Prince (Italian: Il Principe [il ˈprintʃip...
6,Through the Looking-Glass,"*Through the Looking-Glass, and What Alice Fou..."
7,The Wind in the Willows,"The adventures of four amiable animals, Rat, T..."
8,Five Children and It,Haven't you ever thought what you would wish f...
9,The Princess and the Goblin,There was once a little princess whose father ...


## Por hacer:
- [X] publishers eliminar
- [X] Cambiar géneros a inglés
- [X] Añadir años de publicación para los dos libros que están en año 0
- [X] Sacar sinopsis 
- [X] décadas: 
1400-1499, 
1500-1599, 
1600-1699, 
1700-1799, 
1800-1849, 
1850-1899, 
1900-1949, 
1950-1969, 
1970-1989, 
1990-1999, 
2000-2009, 
2010-2019, 
2020-2026

- [X] IRENE: csv titulo y sinopsis ('books_title_synopsis.csv')

- [X] CLARA: csv sin duplicados pero con los años ajustados y demás, csv con duplicados ('books_original.csv' -> con duplicados y 'books_clean.csv' -> sin duplicados)